In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Sambhajinagar_Dataset_V3_Final.csv")

print("Dataset loaded successfully")
print("Original shape:", df.shape)

Dataset loaded successfully
Original shape: (9893, 10)


In [2]:
# Remove metadata columns that are not useful for analysis or modeling
columns_to_remove = ["system:index", ".geo"]

df_clean = df.drop(columns=columns_to_remove)

print("Columns removed:", columns_to_remove)
print("Cleaned shape:", df_clean.shape)
print("\nRemaining columns:")
print(df_clean.columns.tolist())

Columns removed: ['system:index', '.geo']
Cleaned shape: (9893, 8)

Remaining columns:
['Elevation', 'LST', 'LandCover', 'Latitude', 'Longitude', 'NDBI', 'NDVI', 'Population']


In [3]:
print("Missing values per column:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:")
print(df_clean.duplicated().sum())

Missing values per column:
Elevation     0
LST           0
LandCover     0
Latitude      0
Longitude     0
NDBI          0
NDVI          0
Population    0
dtype: int64

Duplicate rows:
0


In [4]:
# Keep LandCover as an integer category code
df_clean["LandCover"] = df_clean["LandCover"].astype(int)

# Ensure numeric columns are numeric
numeric_columns = [
    "Elevation",
    "LST",
    "Latitude",
    "Longitude",
    "NDBI",
    "NDVI",
    "Population"
]

df_clean[numeric_columns] = df_clean[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

print(df_clean.dtypes)

Elevation       int64
LST           float64
LandCover       int64
Latitude      float64
Longitude     float64
NDBI          float64
NDVI          float64
Population    float64
dtype: object


In [5]:
print("Missing values after type conversion:")
print(df_clean.isnull().sum())

print("\nCleaned dataset shape:", df_clean.shape)

Missing values after type conversion:
Elevation     0
LST           0
LandCover     0
Latitude      0
Longitude     0
NDBI          0
NDVI          0
Population    0
dtype: int64

Cleaned dataset shape: (9893, 8)


In [6]:
expected_landcover_classes = [10, 20, 30, 40, 50, 60, 80]

actual_landcover_classes = sorted(df_clean["LandCover"].unique())

print("Actual LandCover classes:", actual_landcover_classes)
print("Expected LandCover classes:", expected_landcover_classes)

invalid_classes = set(actual_landcover_classes) - set(expected_landcover_classes)

if len(invalid_classes) == 0:
    print("\nLandCover validation passed.")
else:
    print("\nUnexpected LandCover classes found:", invalid_classes)

Actual LandCover classes: [np.int64(10), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(60), np.int64(80)]
Expected LandCover classes: [10, 20, 30, 40, 50, 60, 80]

LandCover validation passed.


In [7]:
range_summary = df_clean[
    [
        "NDVI",
        "NDBI",
        "LST",
        "Elevation",
        "Population"
    ]
].describe().T

range_summary

,count,mean,std,min,25%,50%,75%,max
NDVI,9893.0,0.241080,0.115880,-0.128937,0.160441,0.211398,0.296107,0.751738
NDBI,9893.0,0.029918,0.098665,-0.433368,-0.021819,0.053535,0.099973,0.401310
LST,9893.0,43.463630,3.229270,27.762190,41.241152,43.606422,45.966564,51.956644
Elevation,9893.0,621.370262,85.589970,483.000000,556.000000,597.000000,691.000000,926.000000
Population,9893.0,12.378498,29.300305,0.667908,2.063318,3.155722,7.110859,230.774673


In [8]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Original Columns",
        "Cleaned Rows",
        "Cleaned Columns",
        "Removed Columns",
        "Missing Values After Cleaning",
        "Duplicate Rows After Cleaning",
        "LandCover Classes"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        df_clean.shape[0],
        df_clean.shape[1],
        ", ".join(columns_to_remove),
        int(df_clean.isnull().sum().sum()),
        int(df_clean.duplicated().sum()),
        ", ".join(map(str, actual_landcover_classes))
    ]
})

cleaning_summary

,Metric,Value
0,Original Rows,9893
1,Original Columns,10
2,Cleaned Rows,9893
3,Cleaned Columns,8
4,Removed Columns,"system:index, .geo"
5,Missing Values After Cleaning,0
6,Duplicate Rows After Cleaning,0
7,LandCover Classes,"10, 20, 30, 40, 50, 60, 80"


In [10]:
import os

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../outputs/reports", exist_ok=True)

df_clean.to_csv(
    "../data/processed/cleaned_uhi_v3.csv",
    index=False
)

cleaning_summary.to_csv(
    "../outputs/reports/data_cleaning_summary_v3.csv",
    index=False
)

print("Saved cleaned dataset:")
print("data/processed/cleaned_uhi_v3.csv")

print("\nSaved cleaning summary:")
print("outputs/reports/data_cleaning_summary_v3.csv")

Saved cleaned dataset:
data/processed/cleaned_uhi_v3.csv

Saved cleaning summary:
outputs/reports/data_cleaning_summary_v3.csv


In [11]:
print("Final cleaned dataset preview:")
display(df_clean.head())

print("\nFinal shape:", df_clean.shape)

Final cleaned dataset preview:


,Elevation,LST,LandCover,Latitude,Longitude,NDBI,NDVI,Population
0,919,43.919170,40,19.970832,75.438700,0.127207,0.162286,1.562392
1,488,39.060455,40,19.729093,75.209548,-0.272423,0.570751,1.361367
2,505,40.231127,30,19.788775,75.225639,0.055877,0.099371,15.386780
3,757,44.884761,40,19.960297,75.544582,0.121404,0.196037,2.643492
4,586,48.502735,30,19.867709,75.486062,0.133151,0.165262,31.397209



Final shape: (9893, 8)


## Dataset V3 Cleaning Conclusion

Dataset V3 was cleaned by removing the `system:index` and `.geo` metadata fields. No missing values or duplicate rows were found.

The cleaned dataset contains 9,893 rows and 8 columns: Elevation, LST, LandCover, Latitude, Longitude, NDBI, NDVI, and Population.

`LandCover` was retained as a categorical class-code feature. It will be one-hot encoded during feature engineering. Latitude and Longitude are retained only for mapping and spatial block cross-validation and will not be used as prediction features.